In [1]:
import pandas as pd
import numpy as np

#import yaml
from pathlib import Path

In [2]:
config_path = Path.cwd().parent / "config.yaml"
with open(config_path, "r") as f:
    config = yaml.safe_load(f)

root = Path(config['project_root'])

In [2]:
df = pd.read_parquet("../data/processed/air_quality_weather.parquet")
df.head()

,Date,station,Town,Province,Latitude,Longitude,NO2,PM10,PM2.5,SO2,Temperature,Humidity,Precipitation,WindSpeed,WindDirection
0,2015-01-01,ALGORTA_BBIZI2,Getxo,Bizkaia,43.362056,-3.022782,47.0,25.0,28.0,9.0,6.4,85,0.0,8.0,176
1,2015-01-02,ALGORTA_BBIZI2,Getxo,Bizkaia,43.362056,-3.022782,56.0,24.0,18.0,8.0,8.3,81,0.0,8.9,207
2,2015-01-03,ALGORTA_BBIZI2,Getxo,Bizkaia,43.362056,-3.022782,48.0,33.0,21.0,8.0,8.9,80,0.0,13.0,215
3,2015-01-04,ALGORTA_BBIZI2,Getxo,Bizkaia,43.362056,-3.022782,43.0,31.0,23.0,7.0,9.6,88,0.0,8.7,219
4,2015-01-05,ALGORTA_BBIZI2,Getxo,Bizkaia,43.362056,-3.022782,29.0,18.0,11.0,5.0,9.0,85,0.0,13.9,170


## Temporal Feature Engineering

In [3]:
df["year"] = df["Date"].dt.year
df["month"] = df["Date"].dt.month
df["day"] = df["Date"].dt.day
df["day_of_year"] = df["Date"].dt.dayofyear
df["week_of_year"] = df["Date"].dt.isocalendar().week
df["day_of_week"] = df["Date"].dt.dayofweek
season_map = {
    12:"Winter",
    1:"Winter",
    2:"Winter",
    3:"Spring",
    4:"Spring",
    5:"Spring",
    6:"Summer",
    7:"Summer",
    8:"Summer",
    9:"Autumn",
    10:"Autumn",
    11:"Autumn"
}

df["season"] = df["month"].map(season_map)
df.head()

,Date,station,Town,Province,Latitude,Longitude,NO2,PM10,PM2.5,SO2,...,Precipitation,WindSpeed,WindDirection,year,month,day,day_of_year,week_of_year,day_of_week,season
0,2015-01-01,ALGORTA_BBIZI2,Getxo,Bizkaia,43.362056,-3.022782,47.0,25.0,28.0,9.0,...,0.0,8.0,176,2015,1,1,1,1,3,Winter
1,2015-01-02,ALGORTA_BBIZI2,Getxo,Bizkaia,43.362056,-3.022782,56.0,24.0,18.0,8.0,...,0.0,8.9,207,2015,1,2,2,1,4,Winter
2,2015-01-03,ALGORTA_BBIZI2,Getxo,Bizkaia,43.362056,-3.022782,48.0,33.0,21.0,8.0,...,0.0,13.0,215,2015,1,3,3,1,5,Winter
3,2015-01-04,ALGORTA_BBIZI2,Getxo,Bizkaia,43.362056,-3.022782,43.0,31.0,23.0,7.0,...,0.0,8.7,219,2015,1,4,4,1,6,Winter
4,2015-01-05,ALGORTA_BBIZI2,Getxo,Bizkaia,43.362056,-3.022782,29.0,18.0,11.0,5.0,...,0.0,13.9,170,2015,1,5,5,2,0,Winter


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 29008 entries, 0 to 29007
Data columns (total 22 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   Date           29008 non-null  datetime64[us]
 1   station        29008 non-null  str           
 2   Town           29008 non-null  str           
 3   Province       29008 non-null  str           
 4   Latitude       29008 non-null  float64       
 5   Longitude      29008 non-null  float64       
 6   NO2            29008 non-null  float64       
 7   PM10           29008 non-null  float64       
 8   PM2.5          29008 non-null  float64       
 9   SO2            29008 non-null  float64       
 10  Temperature    29008 non-null  float64       
 11  Humidity       29008 non-null  int64         
 12  Precipitation  29008 non-null  float64       
 13  WindSpeed      29008 non-null  float64       
 14  WindDirection  29008 non-null  int64         
 15  year           29008 non-null 

## Lag Features

In [5]:
lags = [1,3,7,14,30,90,365]

for lag in lags:
    df[f"PM25_lag_{lag}"] = (
        df.groupby("station")["PM2.5"]
        .shift(lag)
    )

In [6]:
for lag in lags:
    df[f"PM10_lag_{lag}"] = (
        df.groupby("station")["PM10"]
        .shift(lag)
    )

In [7]:
for lag in lags:
    df[f"NO2_lag_{lag}"] = (
        df.groupby("station")["NO2"]
        .shift(lag)
    )

In [8]:
for lag in lags:
    df[f"SO2_lag_{lag}"] = (
        df.groupby("station")["SO2"]
        .shift(lag)
    )

## Rolling Features

In [9]:
windows = [7,14,30,90,365]

for w in windows:
    df[f"PM25_roll_mean_{w}"] = (
        df.groupby("station")["PM2.5"]
        .transform(
            lambda x:
            x.shift(1).rolling(w).mean()
        )
    )

In [10]:
windows = [7,14,30,90,365]

for w in windows:
    df[f"PM10_roll_mean_{w}"] = (
        df.groupby("station")["PM10"]
        .transform(
            lambda x:
            x.shift(1).rolling(w).mean()
        )
    )

In [11]:
windows = [7,14,30,90,365]

for w in windows:
    df[f"NO2_roll_mean_{w}"] = (
        df.groupby("station")["NO2"]
        .transform(
            lambda x:
            x.shift(1).rolling(w).mean()
        )
    )

In [12]:
windows = [7,14,30,90,365]

for w in windows:
    df[f"SO2_roll_mean_{w}"] = (
        df.groupby("station")["SO2"]
        .transform(
            lambda x:
            x.shift(1).rolling(w).mean()
        )
    )

### Forecast 

In [13]:
df["target_PM25"] = (
    df.groupby("station")["PM2.5"]
    .shift(-1)
)

In [14]:
"""df["target_PM25"] = (
    df.groupby("station")["PM2.5"]
    .shift(-7)
)"""

'df["target_PM25"] = (\n    df.groupby("station")["PM2.5"]\n    .shift(-7)\n)'

In [15]:
df

,Date,station,Town,Province,Latitude,Longitude,NO2,PM10,PM2.5,SO2,...,NO2_roll_mean_14,NO2_roll_mean_30,NO2_roll_mean_90,NO2_roll_mean_365,SO2_roll_mean_7,SO2_roll_mean_14,SO2_roll_mean_30,SO2_roll_mean_90,SO2_roll_mean_365,target_PM25
0,2015-01-01,ALGORTA_BBIZI2,Getxo,Bizkaia,43.362056,-3.022782,47.0,25.0,28.0,9.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,18.0
1,2015-01-02,ALGORTA_BBIZI2,Getxo,Bizkaia,43.362056,-3.022782,56.0,24.0,18.0,8.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,21.0
2,2015-01-03,ALGORTA_BBIZI2,Getxo,Bizkaia,43.362056,-3.022782,48.0,33.0,21.0,8.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,23.0
3,2015-01-04,ALGORTA_BBIZI2,Getxo,Bizkaia,43.362056,-3.022782,43.0,31.0,23.0,7.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,11.0
4,2015-01-05,ALGORTA_BBIZI2,Getxo,Bizkaia,43.362056,-3.022782,29.0,18.0,11.0,5.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29003,2026-05-02,SANTURTZI,Santurtzi,Bizkaia,43.333012,-3.042560,16.0,13.0,11.0,3.0,...,15.571429,15.933333,15.744444,15.113077,3.000000,3.071429,3.533333,4.388889,4.356164,6.0
29004,2026-05-03,SANTURTZI,Santurtzi,Bizkaia,43.333012,-3.042560,12.0,9.0,6.0,3.0,...,15.785714,16.266667,15.811111,15.110337,3.000000,3.071429,3.533333,4.377778,4.353425,3.0
29005,2026-05-04,SANTURTZI,Santurtzi,Bizkaia,43.333012,-3.042560,10.0,5.0,3.0,3.0,...,15.785714,16.500000,15.855556,15.118556,3.000000,3.071429,3.500000,4.377778,4.353425,4.0
29006,2026-05-05,SANTURTZI,Santurtzi,Bizkaia,43.333012,-3.042560,14.0,7.0,4.0,4.0,...,15.500000,16.266667,15.900000,15.115817,3.000000,3.071429,3.433333,4.377778,4.353425,3.0


In [16]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 29008 entries, 0 to 29007
Data columns (total 71 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   Date                29008 non-null  datetime64[us]
 1   station             29008 non-null  str           
 2   Town                29008 non-null  str           
 3   Province            29008 non-null  str           
 4   Latitude            29008 non-null  float64       
 5   Longitude           29008 non-null  float64       
 6   NO2                 29008 non-null  float64       
 7   PM10                29008 non-null  float64       
 8   PM2.5               29008 non-null  float64       
 9   SO2                 29008 non-null  float64       
 10  Temperature         29008 non-null  float64       
 11  Humidity            29008 non-null  int64         
 12  Precipitation       29008 non-null  float64       
 13  WindSpeed           29008 non-null  float64       
 14  W

In [19]:
df.to_parquet("../data/processed/forecasting_dataset.parquet", index=False)

print("✅ Success: forecasting_dataset.parquet saved.")

✅ Success: forecasting_dataset.parquet saved.
